# Comparing --session vs inline auth for repeated API calls

> L3 notebook — exploring when to persist credentials and when to pass them fresh.


## Purpose

When calling the same API repeatedly, HTTPie offers two main paths for authentication: paste the header on every invocation, or persist it in a session file. This notebook walks through both approaches and flags where each one tends to break down.

## Inline auth — the straightforward way

The quickest way to authenticate is to pass the header directly on the command line. HTTPie sends it exactly as written, and nothing is saved to disk.

In [ ]:
# Inline auth — credentials live only in this command
http GET https://api.example.com/user \
  Authorization:"Bearer sk-abc123xyz" \
  Accept:application/json


## The repetition problem

Running the command multiple times means typing or pasting the token each time. If the token rotates, every command must be updated. There is no shared state between invocations.

## Session auth — let HTTPie remember

The `--session` flag tells HTTPie to write cookies and headers to a JSON file after the first request, then replay them on subsequent requests. The `./` prefix keeps the file in the current directory; without it, HTTPie stores the session under `~/.config/httpie/sessions/<host>/<name>.json` as a named session.

In [ ]:
# First request creates the session file
http --session=./session.json \
  GET https://api.example.com/user \
  Authorization:"Bearer sk-abc123xyz" \
  Accept:application/json

# Subsequent requests reuse the session automatically
http --session=./session.json \
  GET https://api.example.com/dashboard \
  Accept:application/json

In [ ]:
# Inspect the session file to see what HTTPie stored
import json, pathlib

session = json.loads(pathlib.Path("session.json").read_text())
print(json.dumps(session, indent=2))

## Comparing the two

| Aspect | Inline auth | Session auth |
|--------|-------------|--------------|
| Token entry | Every command | Once, then persisted |
| File footprint | None | `session.json` in cwd |
| Cross-host leakage | Impossible | Isolated per host by default |
| Rotation recovery | Edit every command | Delete the session file and re-auth |
| CI friendliness | Easy to script | Must ensure the file is writable |

## What to verify next

One thing to check is what happens when the token expires mid-session. The docs note that stale session state can keep sending an old `Authorization` header after a rotation, so deleting the session JSON and re-authenticating is the recovery step. Another angle is whether setting `domain: null` inside the session JSON is useful for cross-domain redirect chains, or whether it introduces more risk than reward for a single-API workflow.